# ERP Support Operations Dashboard

## Objective

Analyze incident management data to evaluate operational performance, SLA compliance, and support efficiency.

## Dataset

**Incident Management Process Enriched Event Log**

- Records: 141,712
- Variables: 36
- Unique Incidents: 24,918

## Project Scope

This project focuses on:

- Incident lifecycle analysis
- SLA compliance monitoring
- Resolution time evaluation
- Operational performance assessment
- Support process efficiency

## Data Overview

The dataset contains event-level records generated throughout the lifecycle of an incident. Multiple records may belong to the same incident, allowing process-level analysis of status changes, reassignments, resolutions, and closures.

In [58]:
# ==================================================
# 📚 IMPORTACIÓN DE LIBRERÍAS
# ==================================================

# 📂🧹🔍📊 Librerías básicas para exploración y análisis

# Librería para trabajar con DataFrames
import pandas as pd
pd.set_option('display.max_columns', None)

# Librería para carpetas
import os

# Librería para cálculos numéricos y matrices
import numpy as np

# Librerías para visualización
import matplotlib.pyplot as plt      # Más personalizable
import seaborn as sns                # Más moderna y estética


# ==================================================
# 🤖 MACHINE LEARNING - SCIKIT LEARN
# ==================================================

# 📈 Modelos de regresión
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# 🔗 Pipeline (cadena de transformaciones)
from sklearn.pipeline import Pipeline

# ⚖️ Escalado de variables
from sklearn.preprocessing import StandardScaler

# 📏 Métricas de evaluación
from sklearn.metrics import mean_squared_error, r2_score

# ✂️ División de datos
from sklearn.model_selection import train_test_split

# ✅ Validación cruzada
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

# 🛡️ Regresión Ridge para reducir el sobreajuste
from sklearn.linear_model import Ridge

# 🔍 Búsqueda de hiperparámetros
from sklearn.model_selection import GridSearchCV

In [ ]:
#Load information

df = pd.read_csv(
    "../Data/Raw/incident+management+process+enriched+event+log.zip"
)
df_cl=df


## Clean Data

In [60]:
# Replace missing value indicators

df_cl = df_cl.replace("?", np.nan)


# Convert date-related columns to datetime format
df_cl["opened_at"] = pd.to_datetime(df_cl["opened_at"],format="%d/%m/%Y %H:%M")
df_cl["sys_created_at"] = pd.to_datetime(df_cl["sys_created_at"],format="%d/%m/%Y %H:%M")
df_cl["sys_updated_at"] = pd.to_datetime(df_cl["sys_updated_at"],format="%d/%m/%Y %H:%M")
df_cl["resolved_at"] = pd.to_datetime(df_cl["resolved_at"],format="%d/%m/%Y %H:%M")
df_cl["closed_at"] = pd.to_datetime(df_cl["closed_at"],format="%d/%m/%Y %H:%M")


# Convert categorical variables to optimize memory usage
# and reflect that these fields represent discrete
# business categories.

df_cl["incident_state"] = df_cl["incident_state"].astype("category")
df_cl["contact_type"] = df_cl["contact_type"].astype("category")
df_cl["impact"] = df_cl["impact"].astype("category")
df_cl["urgency"] = df_cl["urgency"].astype("category")
df_cl["priority"] = df_cl["priority"].astype("category")
df_cl["notify"] = df_cl["notify"].astype("category")


# Convert sparse reference fields into boolean indicators.
# These variables contain a high proportion of null values,
# therefore only the existence of an associated record is retained.

df_cl["problem_id"] = df_cl["problem_id"].notna()
df_cl["rfc"] = df_cl["rfc"].notna()
df_cl["vendor"] = df_cl["vendor"].notna()
df_cl["caused_by"] = df_cl["caused_by"].notna()


# Remove redundant or low-value columns

# Remove redundant creation timestamp.
# The dataset already contains opened_at, which provides
# equivalent information for incident creation analysis.
df_cl = df_cl.drop (columns=["sys_created_at"])

# Remove related creator field after removing
# the associated creation timestamp.
df_cl = df_cl.drop (columns=["sys_created_by"])

# Remove configuration item identifier because it does not
# support the analytical objectives of this project.
df_cl = df_cl.drop (columns=["cmdb_ci"])

## Initial Dataset Assessment Summary

The dataset contains:

- 141,712 records
- 36 variables

All variables were reviewed and classified according
to their business relevance, data quality and analytical
value.

The detailed data dictionary was documented separately.

Summary of findings:

| Category | Number of Variables |
|----------|----------|
| Identifiers | 12 |
| Date Fields | 5 |
| Categorical Variables | 6 |
| Boolean Variables | 7 |
| Operational Metrics | 6 |

Key observations:

- Several identifiers are repeated because multiple rows belong to the same incident.
- Priority, impact and urgency are core analytical variables.
- Some reference fields contain a high proportion of null values.
- Redundant timestamp fields were identified.

Desglose de la tabla
- Columnas: 36
- Filas: 141.712

|Columna|	Descripcion|
|----------|----------|
|number|	ID del caso|
|incident_state|	Refiere variable categorica del estado de un incidente|
|active|	Indica si un caso esta activo|
|reassignment_count|	Conteo de reasignaciones|
|reopen_count|	Conteo de devoluciones|
|sys_mod_count|	Conteo de cambio de estado|
|made_sla|	Indica si se supero el SLA|
|caller_id|	Indica el id del usuario afectado|
|opened_by|	Indica el id del usuario reporta|
|opened_at|	Fecha y hora de apertura del caso |
|sys_created_by|	Indica el id del usuario que creo el incidente |
|sys_created_at|	Fecha y hora de creacion del caso |
|sys_updated_by|	idica el id del usuario que actualizó el incidente y generó el registro de registro actual|
|sys_updated_at|	Fecha y hora de la actualizacion del incidente|
|contact_type|	Refiere variable categorica del tipo de contacto|
|location|	Indica Id de la ubiacacion|
|category|	Indica Id de la categoria|
|subcategory|	Indica Id de la subcategoria, esta relacionado con la categoria|
|u_symptom|	Indcia id del u_smyptom, refiere al codigo de percepcion del sintoma por parte del usuario|
|cmdb_ci|	Id del (elemento de confirmación) identificador utilizado para informar del artículo afectado (no obligatorio)|
|impact|	Refiere variable categorica del impacto del incidente|
|urgency|	Refiere variable categorica de la urgencia del incidente|
|priority|	Variable categorica calculada por el sistema en función de 'impacto' y 'urgencia';|
|assignment_group|	id del del grupo de apoyo responsable del incidente|
|assigned_to|	Id del usuario resposanble del incidente|
|knowledge|	atributo booleano que muestra si se utilizó un documento de base de conocimiento para resolver el incidente|
|u_priority_confirmation|	atributo booleano que indica si el campo de prioridad ha sido comprobado dos veces|
|notify|	Variable categorica que muestra el tipo de notifiacion|
|problem_id|	Indica el Id del problema asociado con el incideente|
|rfc|	Identificador de la solicitud de cambio asociada al incidente|
|vendor|	Identificador del proveedor responsable del incidente|
|caused_by|	identificador del RFC responsable del incidente|
|closed_code|	identificador de la resolución del incidente|
|resolved_by|	identificador del usuario que resolvió el incidente|
|resolved_at|	fecha y hora de resolución del usuario incidente (variable dependiente)|
|closed_at|	fecha y hora de cierre del usuario incidente (variable dependiente)|
